# Notebook 02: Data Cleaning and Feature Preparation

## Purpose
This notebook applies filtering, standardization, and feature engineering to produce a clean station-level dataset for downstream joins and aggregation.

## Cleaning goals
- keep only active, public, in-scope stations
- normalize critical fields such as ZIP and dates
- construct analysis-ready features used in later notebooks

In [2]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

from pyspark.sql import SparkSession

# Keep a dedicated Spark app name so cleaning-stage jobs are easy to spot in logs.
spark = SparkSession.builder \
    .appName("EV_Charging_Cleaning") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 11:34:01 WARN Utils: Your hostname, Krishs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.228 instead (on interface en0)
26/05/03 11:34:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 11:34:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 11:34:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.1


In [3]:
import os
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, DateType

BASE_DIR = os.getcwd()
AFDC_PATH = os.path.join(BASE_DIR, "alt_fuel_stations.csv")
CENSUS_PATH = os.path.join(BASE_DIR, "acs_zcta_combined.csv")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

key_columns = [
    "ID", "Station Name", "City", "State", "ZIP",
    "Latitude", "Longitude", "Access Code", "Status Code",
    "Open Date", "EV Level1 EVSE Num", "EV Level2 EVSE Num",
    "EV DC Fast Count", "EV Network", "EV Connector Types", "Country"
]

df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv(AFDC_PATH) \
    .select(key_columns)

print("Raw row count:", df_raw.count())

Raw row count: 85659


In [4]:
# This is our first and most impactful filter
# We are removing: private stations, planned/temp stations
# These don't represent real public charging infrastructure

df_filtered = df_raw.filter(
    (F.col("Status Code") == "E") &       # E = Open/Available
    (F.col("Access Code") == "public") &   # public only
    (F.col("Country") == "US")             # US only
)

removed = df_raw.count() - df_filtered.count()
print(f"Rows before filter: {df_raw.count()}")
print(f"Rows after filter:  {df_filtered.count()}")
print(f"Rows removed:       {removed}")

Rows before filter: 85659
Rows after filter:  78127
Rows removed:       7532


In [5]:
# Only 6 rows affected but we drop them because
# every downstream analysis needs city and ZIP

df_no_nulls = df_filtered.dropna(subset=["City", "ZIP"])

print(f"Rows after dropping null City/ZIP: {df_no_nulls.count()}")

Rows after dropping null City/ZIP: 78121


In [6]:
# Null in a charger count column means that TYPE of charger
# doesn't exist at this station - not that data is missing
# So null = 0 is the correct interpretation here

df_filled = df_no_nulls.fillna({
    "EV Level1 EVSE Num": 0,
    "EV Level2 EVSE Num": 0,
    "EV DC Fast Count": 0
})

print("Null check after filling:")
for col_name in ["EV Level1 EVSE Num", "EV Level2 EVSE Num", "EV DC Fast Count"]:
    null_count = df_filled.filter(F.col(col_name).isNull()).count()
    print(f"  {col_name}: {null_count} nulls remaining")

Null check after filling:
  EV Level1 EVSE Num: 0 nulls remaining
  EV Level2 EVSE Num: 0 nulls remaining
  EV DC Fast Count: 0 nulls remaining


In [7]:
# After filling nulls, any station where ALL counts are 0
# is genuinely useless - it has no chargers at all
# This should remove those 17 problematic rows we found

df_valid = df_filled.filter(
    (F.col("EV Level1 EVSE Num") + 
     F.col("EV Level2 EVSE Num") + 
     F.col("EV DC Fast Count")) > 0
)

print(f"Rows after removing zero-charger stations: {df_valid.count()}")

Rows after removing zero-charger stations: 78120


In [8]:
# ZIP codes like "601" should be "00601"
# lpad() pads on the left with "0" until the string is 5 characters long
# This is critical for matching with Census ZCTA data

df_zip = df_valid.withColumn(
    "ZIP",
    F.lpad(F.col("ZIP").cast("string"), 5, "0")
)

# Verify the fix worked
print("=== ZIP length distribution after fix ===")
df_zip.withColumn("zip_length", F.length(F.col("ZIP"))) \
    .groupBy("zip_length").count() \
    .orderBy("zip_length") \
    .show()

=== ZIP length distribution after fix ===
+----------+-----+
|zip_length|count|
+----------+-----+
|         5|78120|
+----------+-----+



In [9]:
# Convert the string "1999-10-15" to an actual DateType
# Then extract year and month as separate integer columns
# Vandana needs install_year and install_month directly

df_dates = df_zip.withColumn(
    "open_date_parsed",
    F.to_date(F.col("Open Date"), "yyyy-MM-dd")
).withColumn(
    "install_year",
    F.year(F.col("open_date_parsed"))
).withColumn(
    "install_month",
    F.month(F.col("open_date_parsed"))
)

# Verify - check how many nulls remain in parsed date
null_dates = df_dates.filter(F.col("open_date_parsed").isNull()).count()
print(f"Stations with unparseable/null open date: {null_dates}")
print(f"These will be excluded from time-based analysis only")

# Show sample
df_dates.select("Open Date", "open_date_parsed", "install_year", "install_month") \
    .show(5)

Stations with unparseable/null open date: 234
These will be excluded from time-based analysis only
+----------+----------------+------------+-------------+
| Open Date|open_date_parsed|install_year|install_month|
+----------+----------------+------------+-------------+
|1995-08-30|      1995-08-30|        1995|            8|
|1997-07-30|      1997-07-30|        1997|            7|
|2012-12-11|      2012-12-11|        2012|           12|
|1997-08-30|      1997-08-30|        1997|            8|
|2014-01-20|      2014-01-20|        2014|            1|
+----------+----------------+------------+-------------+
only showing top 5 rows


In [10]:
# Trim whitespace and standardize casing
# This prevents groupBy failures where "CA" and "Ca" 
# would be counted as different states

df_standardized = df_dates \
    .withColumn("State", F.trim(F.upper(F.col("State")))) \
    .withColumn("City", F.trim(F.initcap(F.col("City")))) \
    .withColumn("EV Network", F.trim(F.col("EV Network")))

# Verify state standardization
print("=== Sample states after standardization ===")
df_standardized.select("State").distinct().orderBy("State").show(10)

=== Sample states after standardization ===
+-----+
|State|
+-----+
|   AK|
|   AL|
|   AR|
|   AZ|
|   CA|
|   CO|
|   CT|
|   DC|
|   DE|
|   FL|
+-----+
only showing top 10 rows


## Feature Engineering Layer

With core cleaning complete, this section derives analysis fields such as `total_ports`, `charger_level`, `is_dcfc`, and `region`.
These variables support equity metrics, accessibility classification, and regional comparisons.

In [11]:
# total_ports = sum of all charger types at each station
# This is a useful aggregate column for density calculations

df_totals = df_standardized.withColumn(
    "total_ports",
    F.col("EV Level1 EVSE Num") + 
    F.col("EV Level2 EVSE Num") + 
    F.col("EV DC Fast Count")
)

df_totals.select("Station Name", "EV Level1 EVSE Num", 
                  "EV Level2 EVSE Num", "EV DC Fast Count", 
                  "total_ports").show(5)

+--------------------+------------------+------------------+----------------+-----------+
|        Station Name|EV Level1 EVSE Num|EV Level2 EVSE Num|EV DC Fast Count|total_ports|
+--------------------+------------------+------------------+----------------+-----------+
|Los Angeles Conve...|                 0|                 8|               0|          8|
|Scripps Green Hos...|                 0|                 1|               0|          1|
|       Galpin Motors|                 0|                 2|               0|          2|
|   Galleria at Tyler|                 0|                 4|               0|          4|
|City of Pasadena ...|                 0|                26|               0|         26|
+--------------------+------------------+------------------+----------------+-----------+
only showing top 5 rows


In [12]:
# Classify each station by its PRIMARY charger type
# If it has DCFC it's a fast charging station regardless of what else it has
# If no DCFC but has Level2, it's a Level2 station
# Otherwise Level1

df_level = df_totals.withColumn(
    "charger_level",
    F.when(F.col("EV DC Fast Count") > 0, "DCFC")
     .when(F.col("EV Level2 EVSE Num") > 0, "Level2")
     .otherwise("Level1")
)

print("=== Charger level distribution ===")
df_level.groupBy("charger_level").count() \
    .orderBy("count", ascending=False).show()

=== Charger level distribution ===
+-------------+-----+
|charger_level|count|
+-------------+-----+
|       Level2|63543|
|         DCFC|14488|
|       Level1|   89|
+-------------+-----+



In [13]:
# Binary flag - 1 if station has any DCFC ports, 0 otherwise
# Riddhi needs this for the charging desert classification model

df_flag = df_level.withColumn(
    "is_dcfc",
    F.when(F.col("EV DC Fast Count") > 0, 1).otherwise(0)
)

dcfc_count = df_flag.filter(F.col("is_dcfc") == 1).count()
total = df_flag.count()
print(f"DCFC stations: {dcfc_count} ({dcfc_count/total*100:.1f}% of all public open stations)")

DCFC stations: 14488 (18.5% of all public open stations)


In [14]:
# Map each US state to its Census region
# This is a hardcoded lookup - there's no formula for this
# Vandana needs this for regional comparison charts

region_map = {
    # Northeast
    "CT": "Northeast", "ME": "Northeast", "MA": "Northeast",
    "NH": "Northeast", "RI": "Northeast", "VT": "Northeast",
    "NJ": "Northeast", "NY": "Northeast", "PA": "Northeast",
    # Midwest
    "IL": "Midwest", "IN": "Midwest", "MI": "Midwest",
    "OH": "Midwest", "WI": "Midwest", "IA": "Midwest",
    "KS": "Midwest", "MN": "Midwest", "MO": "Midwest",
    "NE": "Midwest", "ND": "Midwest", "SD": "Midwest",
    # South
    "DE": "South", "FL": "South", "GA": "South",
    "MD": "South", "NC": "South", "SC": "South",
    "VA": "South", "DC": "South", "WV": "South",
    "AL": "South", "KY": "South", "MS": "South",
    "TN": "South", "AR": "South", "LA": "South",
    "OK": "South", "TX": "South",
    # West
    "AZ": "West", "CO": "West", "ID": "West",
    "MT": "West", "NV": "West", "NM": "West",
    "UT": "West", "WY": "West", "AK": "West",
    "CA": "West", "HI": "West", "OR": "West", "WA": "West"
}

# Convert the Python dictionary to a Spark map literal
mapping_expr = F.create_map(
    [F.lit(x) for pair in region_map.items() for x in pair]
)

df_region = df_flag.withColumn(
    "region",
    mapping_expr[F.col("State")]
)

print("=== Region distribution ===")
df_region.groupBy("region").count().orderBy("count", ascending=False).show()

# Check for any states that didn't get mapped (would show as null)
unmapped = df_region.filter(F.col("region").isNull())
print(f"Stations with unmapped region: {unmapped.count()}")
if unmapped.count() > 0:
    unmapped.select("State").distinct().show()

=== Region distribution ===
+---------+-----+
|   region|count|
+---------+-----+
|     West|30569|
|    South|20190|
|Northeast|16259|
|  Midwest|11067|
|     NULL|   35|
+---------+-----+

Stations with unmapped region: 35
+-----+
|State|
+-----+
|   PR|
+-----+



In [15]:
# A clean summary of what the dataset looks like after all cleaning

print("=" * 50)
print("CLEANING SUMMARY")
print("=" * 50)
print(f"Original rows:      {df_raw.count():,}")
print(f"Final clean rows:   {df_region.count():,}")
print(f"Rows removed:       {df_raw.count() - df_region.count():,}")
print(f"\nColumns in clean dataset: {len(df_region.columns)}")
print(f"\nNull counts in key columns:")

key_cols = ["State", "City", "ZIP", "Latitude", "Longitude",
            "open_date_parsed", "EV Network", "charger_level", "region"]

for col_name in key_cols:
    n = df_region.filter(F.col(col_name).isNull()).count()
    print(f"  {col_name:<25}: {n}")

CLEANING SUMMARY
Original rows:      85,659
Final clean rows:   78,120
Rows removed:       7,539

Columns in clean dataset: 23

Null counts in key columns:
  State                    : 0
  City                     : 0
  ZIP                      : 0
  Latitude                 : 0
  Longitude                : 0
  open_date_parsed         : 234
  EV Network               : 0
  charger_level            : 0
  region                   : 35


In [17]:
# Filter out Puerto Rico - outside scope of NEVI/state-level analysis
df_final = df_region.filter(F.col("State") != "PR")

print(f"Rows after removing PR: {df_final.count()}")
print(f"Null regions remaining: {df_final.filter(F.col('region').isNull()).count()}")

Rows after removing PR: 78085
Null regions remaining: 0


In [18]:
CLEAN_STATIONS_PATH = os.path.join(OUTPUT_DIR, "cleaned_stations.parquet")

# Overwrite ensures reruns stay deterministic and do not append duplicate records.
df_final.write \
    .mode("overwrite") \
    .parquet(CLEAN_STATIONS_PATH)

print(f"Saved cleaned stations to: {CLEAN_STATIONS_PATH}")

verify = spark.read.parquet(CLEAN_STATIONS_PATH)
print(f"Verification read - row count: {verify.count()}")
print(f"Columns: {verify.columns}")

[Stage 105:>                                                        (0 + 1) / 1]

Saved cleaned stations to: /Users/krishjani/Documents/Sem 2/Big Data/EV Infrastructure Equity Analyzer/output/cleaned_stations.parquet
Verification read - row count: 78085
Columns: ['ID', 'Station Name', 'City', 'State', 'ZIP', 'Latitude', 'Longitude', 'Access Code', 'Status Code', 'Open Date', 'EV Level1 EVSE Num', 'EV Level2 EVSE Num', 'EV DC Fast Count', 'EV Network', 'EV Connector Types', 'Country', 'open_date_parsed', 'install_year', 'install_month', 'total_ports', 'charger_level', 'is_dcfc', 'region']


In [ ]:
## Notebook 02 Output

The cleaned output is persisted to `output/cleaned_stations.parquet` and contains a validated station-level feature set for joining with Census data in Notebook 03.